***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.10 线性代数：从测量方程到矩阵表示](2_10_linear_algebra.ipynb)
    * 下一节：[2.12 补充专题：立体角与天球面积元素](2_12_solid_angle.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.11 最小二乘与参数估计<a id='groundwork:sec:leastsquares'></a>

最小二乘回答的是一个普遍问题：给定带噪数据和一个参数化模型，怎样选择参数才能让模型与数据尽可能一致。射电干涉测量中的波束拟合、增益求解、源模型拟合、自校准和许多成像迭代，都可以看成最小二乘或其变体。

第 2.10 节已经给出加权正规方程。本节进一步解释残差和权重的统计含义，区分线性与非线性最小二乘，并说明 Gauss-Newton 与 Levenberg-Marquardt 方法如何通过局部线性化求解非线性问题。


### 2.11.1 残差、权重与似然<a id='math:sec:ls_residual_weight'></a>

设数据向量为 $\mathbf{d}$，模型预测为 $\mathbf{m}(\mathbf{x})$，参数向量为 $\mathbf{x}$。残差定义为

<a id='math:eq:ls_residual'></a><!--\label{math:eq:ls_residual}-->
$$
\mathbf{r}(\mathbf{x})=\mathbf{d}-\mathbf{m}(\mathbf{x}).
$$

若所有数据点具有相同方差且相互独立，最自然的目标函数是残差平方和

$$
\|\mathbf{r}\|_2^2=\sum_i r_i^2.
$$

真实观测通常有不同噪声水平，甚至存在协方差结构。若噪声服从零均值高斯分布，协方差为 $\boldsymbol{\Sigma}$，则负对数似然中与参数相关的部分为

<a id='math:eq:weighted_ls_objective'></a><!--\label{math:eq:weighted_ls_objective}-->
$$
\chi^2(\mathbf{x})=\mathbf{r}^H(\mathbf{x})\mathbf{W}\mathbf{r}(\mathbf{x}),
\qquad
\mathbf{W}=\boldsymbol{\Sigma}^{-1}.
$$

因此，加权最小二乘不是任意引入的技巧，而是高斯噪声模型下的最大似然估计。权重越大，表示相应数据越可靠，对解的约束越强。


从几何上看，线性最小二乘是在数据空间中把观测向量投影到模型子空间上。若模型为 $\mathbf{m}=\mathbf{A}\mathbf{x}$，所有可能模型组成 $\mathbf{A}$ 的列空间。最优模型 $\mathbf{A}\hat{\mathbf{x}}$ 是离数据 $\mathbf{d}$ 最近的列空间点，残差与列空间正交。

![最小二乘几何](figures/least_squares_projection_geometry.png)

**图 2.11.1** 最小二乘的投影解释。最优残差与模型子空间正交，这一正交条件正是正规方程的几何来源。


### 2.11.2 线性最小二乘与正规方程<a id='math:sec:linear_least_squares'></a>

线性模型写作

$$
\mathbf{d}=\mathbf{A}\mathbf{x}+\boldsymbol{\epsilon}.
$$

最小化

$$
\chi^2=(\mathbf{d}-\mathbf{A}\mathbf{x})^H\mathbf{W}(\mathbf{d}-\mathbf{A}\mathbf{x})
$$

得到正规方程

<a id='math:eq:ls_normal_equations'></a><!--\label{math:eq:ls_normal_equations}-->
$$
\mathbf{A}^H\mathbf{W}\mathbf{A}\hat{\mathbf{x}}
=\mathbf{A}^H\mathbf{W}\mathbf{d}.
$$

形式解为

$$
\hat{\mathbf{x}}=(\mathbf{A}^H\mathbf{W}\mathbf{A})^{-1}\mathbf{A}^H\mathbf{W}\mathbf{d}.
$$

但数值计算中应避免显式求逆。更稳定的做法是使用 QR 分解、Cholesky 分解、SVD 或迭代方法直接解线性系统。当 $\mathbf{A}$ 的列近似线性相关，或某些参数方向几乎没有被数据约束时，正规矩阵会病态，解会对噪声极其敏感。


在线性高斯模型中，若权重矩阵正确，参数估计的协方差近似为

<a id='math:eq:ls_parameter_covariance'></a><!--\label{math:eq:ls_parameter_covariance}-->
$$
\mathrm{Cov}(\hat{\mathbf{x}})=(\mathbf{A}^H\mathbf{W}\mathbf{A})^{-1}.
$$

这说明正规矩阵不仅决定最优解，也决定参数不确定度。小奇异值对应弱约束方向；在成像和定标中，这些方向通常需要先验、正则化或更好的观测设计来控制。


### 2.11.3 非线性最小二乘与 Gauss-Newton<a id='math:sec:gauss_newton'></a>

若模型 $\mathbf{m}(\mathbf{x})$ 非线性，就不能一次性写成 $\mathbf{A}\mathbf{x}$。Gauss-Newton 方法的核心，是在当前参数 $\mathbf{x}_k$ 附近做一阶展开：

<a id='math:eq:gn_linearization'></a><!--\label{math:eq:gn_linearization}-->
$$
\mathbf{m}(\mathbf{x}_k+\delta\mathbf{x})
\approx \mathbf{m}(\mathbf{x}_k)+\mathbf{J}\delta\mathbf{x},
$$

其中雅可比矩阵为

$$
J_{ij}=\frac{\partial m_i}{\partial x_j}.
$$

令当前残差为

$$
\mathbf{r}_k=\mathbf{d}-\mathbf{m}(\mathbf{x}_k),
$$

则局部问题变成

$$
\mathbf{r}_k\approx \mathbf{J}\delta\mathbf{x}.
$$

对应的加权正规方程为

<a id='math:eq:gn_update'></a><!--\label{math:eq:gn_update}-->
$$
\mathbf{J}^H\mathbf{W}\mathbf{J}\,\delta\mathbf{x}
=\mathbf{J}^H\mathbf{W}\mathbf{r}_k,
$$

参数更新为

$$
\mathbf{x}_{k+1}=\mathbf{x}_k+\delta\mathbf{x}.
$$

Gauss-Newton 把非线性问题转化为一串局部线性最小二乘问题。它在接近最优解且模型线性化足够准确时收敛很快，但离解太远时可能不稳定。


### 2.11.4 Levenberg-Marquardt 阻尼<a id='math:sec:levenberg_marquardt'></a>

Levenberg-Marquardt（LM）方法在 Gauss-Newton 正规方程中加入阻尼项：

<a id='math:eq:lm_update'></a><!--\label{math:eq:lm_update}-->
$$
(\mathbf{J}^H\mathbf{W}\mathbf{J}+\lambda\mathbf{D})\delta\mathbf{x}
=\mathbf{J}^H\mathbf{W}\mathbf{r}_k.
$$

$\mathbf{D}$ 常取单位矩阵或 $\mathbf{J}^H\mathbf{W}\mathbf{J}$ 的对角部分。阻尼因子 $\lambda$ 控制更新方式：$\lambda$ 很小时，算法接近 Gauss-Newton；$\lambda$ 很大时，更新更像保守的梯度步。实际求解器通常会动态调节 $\lambda$：若一步更新降低了 $\chi^2$，减小阻尼；若更新使目标函数变差，增大阻尼并尝试更保守的步长。

LM 的优势是稳健，但它仍然是局部方法。初值、参数尺度、雅可比质量、权重设置和退化方向都会影响收敛路径。


### 2.11.5 非线性正弦拟合示例<a id='math:sec:lm_sinusoid_example'></a>

考虑模型

<a id='math:eq:sinusoid_model'></a><!--\label{math:eq:sinusoid_model}-->
$$
m_i=x_1\sin(2\pi x_2 t_i+x_3),
$$

其中 $x_1$ 是振幅，$x_2$ 是频率，$x_3$ 是相位。雅可比矩阵的三列分别为

$$
\frac{\partial m_i}{\partial x_1}=\sin(2\pi x_2t_i+x_3),
$$

$$
\frac{\partial m_i}{\partial x_2}=2\pi t_i x_1\cos(2\pi x_2t_i+x_3),
$$

$$
\frac{\partial m_i}{\partial x_3}=x_1\cos(2\pi x_2t_i+x_3).
$$

这个模型足够简单，便于看清“残差、雅可比、阻尼更新”的作用；同时它又是非线性的，频率参数尤其容易表现出初值敏感性。

![LM 收敛路径](figures/least_squares_lm_convergence.png)

**图 2.11.2** LM 在两个初始值下的收敛行为。较好的初值能迅速进入正确吸引域；较差的初值即使降低了残差，也可能停在错误的局部结构附近。


这类初值敏感性在自校准和增益拟合中非常常见。非线性最小二乘不是单纯“解一个方程”，而是在参数空间中寻找目标函数的低谷。若模型存在相位包裹、尺度退化、增益-天空耦合或弱约束方向，求解器可能需要分层求解、先验模型、参数重标度和逐步放宽自由度。


### 2.11.6 与干涉测量的关系<a id='math:sec:least_squares_interferometry'></a>

干涉成像中的许多问题都可以写成最小二乘形式。图像重建可以最小化模型可见度与观测可见度之间的加权残差；增益校准可以最小化校准后基线可见度与天空模型之间的残差；波束拟合和源参数估计也都遵循同一结构。

线性问题的关键是矩阵秩、条件数和权重；非线性问题还要额外面对初值、局部极小值和参数化选择。下一章之后，许多看似复杂的算法都可以回到本节的基本构件：残差、雅可比、权重、正规方程、阻尼和收敛判断。


***

* 下一节：[2.12 补充专题：立体角与天球面积元素](2_12_solid_angle.ipynb)
